In [2]:
import pandas as pd
import os
import glob

In [5]:
# ── 1. Find all pgs_id_list_* files ──────────────────────────────────────────
files = ["pgs_id_list_260217.csv", "pgs_id_list_260410.csv"]

# ── 2. Define target columns (superset) ──────────────────────────────────────
COLS = ["ontology", "pgs_num", "icd", "icd_root", "description",
        "pgs_ids", "pgs_api_num", "pgs_urls"]

# ── 3. Read & normalise each file ────────────────────────────────────────────
dfs = []
for f in files:
    f = os.path.join(os.getcwd(), f)
    df = pd.read_csv(f)
    # Add any missing columns as NaN
    for col in COLS:
        if col not in df.columns:
            df[col] = None
    dfs.append(df[COLS])
    print(f"  {os.path.basename(f)}: {len(df)} rows")

# ── 4. Concatenate ────────────────────────────────────────────────────────────
combined = pd.concat(dfs, ignore_index=True)
print(f"\nBefore dedup: {combined.shape[0]} rows")

# ── 5. Deduplicate on ontology ───────────────────────────────────────────────
combined["_dedup_key"] = combined["ontology"].str.lower().str.strip()
combined.drop_duplicates(subset="_dedup_key", keep="first", inplace=True)
combined.drop(columns="_dedup_key", inplace=True)
combined.reset_index(drop=True, inplace=True)
print(f"After dedup:  {combined.shape[0]} rows")

# ── 6. Save ───────────────────────────────────────────────────────────────────
output_path = os.path.join(os.getcwd(), "pgs_id_list_binary_combined.csv")
combined.to_csv(output_path, index=False)
print(f"\nSaved → {output_path}")

  pgs_id_list_260217.csv: 221 rows
  pgs_id_list_260410.csv: 39 rows

Before dedup: 260 rows
After dedup:  260 rows

Saved → /Users/zhangjiyao/Documents/Upenn/Genetic_Agent/disease_preprocess/pgs_id_list_binary_combined.csv
